# Week 55, Day 1: Binary and bitwise operators

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w055d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
a, b = 0b1100, 0b1010
print(a & b, a | b, a ^ b)
print(bin(a & b), bin(a | b), bin(a ^ b))
print(5 << 3, 100 >> 2, ~5)

In [ ]:
for n in (11, 200, 255, 4096):
    print(n, f"{n:b}", f"{n:o}", f"{n:x}", n.bit_length(), n.bit_count())
print(int("ff", 16), int("0b1011", 0), int("777", 8))

In [ ]:
x = 0b10110
print([i for i in range(x.bit_length()) if x >> i & 1])
print(bin(x | 1 << 0), bin(x & ~(1 << 2)), bin(x ^ 0b11111))
print(x & 3 + 1, (x & 3) + 1, 1 << 3 - 1, (1 << 3) - 1)   # the precedence trap
key = 0b1010_0110
message = 0b0110_0001
secret = message ^ key
print(bin(secret), secret ^ key == message)

In [ ]:
def to_unsigned(x, bits):
    return x & ((1 << bits) - 1)

def to_signed(u, bits):
    return u - (1 << bits) if u >> (bits - 1) & 1 else u

for x in (5, -1, -6, -128, 127):
    u = to_unsigned(x, 8)
    print(x, f"{u:08b}", u, to_signed(u, 8))
print(~5, -5 - 1, bin(~5 & 0xFF))

In [ ]:
def pack_rgb(r, g, b):
    return r << 16 | g << 8 | b

def unpack_rgb(colour):
    return colour >> 16 & 0xFF, colour >> 8 & 0xFF, colour & 0xFF

orange = pack_rgb(255, 136, 0)
print(hex(orange), unpack_rgb(orange), f"#{orange:06X}")

READ, WRITE, EXECUTE = 4, 2, 1
mode = 0o750
for shift, who in ((6, "owner"), (3, "group"), (0, "others")):
    bits = mode >> shift & 0b111
    print(who, "r" if bits & READ else "-", "w" if bits & WRITE else "-", "x" if bits & EXECUTE else "-")

> **Key idea.** Binary digits are worth powers of two; octal groups bits in threes and hex in fours. Read bases with int(text, base), show them with bin, oct, hex or format specifications like 08b and 02X. & keeps bits in both, | in either, ^ in exactly one, ~ flips all, << and >> multiply and floor-divide by powers of two. Test bit i with x >> i & 1; set, clear and flip with |, & ~ and ^. Arithmetic binds tighter than shifts and bitwise operators, so bracket (1 << k) − 1. Python treats negatives as having infinitely many leading 1s, so ~x == −x − 1; mask with (1 << bits) − 1 for fixed widths, and subtract 2^bits when the top bit is set to go back. Pack fields with shifts and |, unpack with shifts and &.

### Exercise: Bases by hand

Write `to_base(n, base)`: the digits of a non-negative whole number n in a base from 2 to 16, as a lowercase string (`"0"` for 0), **without** `bin`, `oct`, `hex` or format specifications: repeatedly take `n % base` and divide. Then `from_base(text, base)`: the reverse, **without** `int(text, base)` (you may use `int` on single digits, or a lookup string).

In [ ]:
def to_base(n, base):
    ___

def from_base(text, base):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_bases():\n    \"bases\"\n    assert to_base(11, 2) == \"1011\" and to_base(255, 16) == \"ff\" and to_base(0, 7) == \"0\" and to_base(64, 8) == \"100\"\n    assert from_base(\"1011\", 2) == 11 and from_base(\"ff\", 16) == 255 and from_base(\"0\", 3) == 0 and from_base(\"FF\", 16) == 255\n    rng = random.Random(1)\n    for _ in range(300):\n        n, b = rng.randint(0, 10 ** 12), rng.randint(2, 16)\n        assert int(to_base(n, b), b) == n and from_base(to_base(n, b), b) == n\n    body = source.split(\"def to_base\")[1]\n    assert \"bin(\" not in body and \"hex(\" not in body and \"oct(\" not in body and \":b}\" not in body and \":x}\" not in body\n    assert \"int(text\" not in source")

### Exercise: File permissions

Write `to_symbolic(mode)`: a permission number like `0o750` as the nine-character string `"rwxr-x---"`, and `from_symbolic(text)`: the reverse. Then `allowed(mode, who, action)`: True if `who` (`"owner"`, `"group"` or `"others"`) may do `action` (`"read"`, `"write"` or `"execute"`). Use shifts and masks; raise `ValueError` for a symbolic string that isn't nine characters of the right letters or `-`.

In [ ]:
def to_symbolic(mode):
    ___

def from_symbolic(text):
    ___

def allowed(mode, who, action):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_perms():\n    \"permissions\"\n    assert to_symbolic(0o750) == \"rwxr-x---\" and to_symbolic(0o644) == \"rw-r--r--\" and to_symbolic(0) == \"---------\"\n    assert from_symbolic(\"rwxr-x---\") == 0o750 and from_symbolic(\"rw-rw-rw-\") == 0o666\n    for mode in range(0o1000):\n        assert from_symbolic(to_symbolic(mode)) == mode\n    assert allowed(0o750, \"group\", \"execute\") and not allowed(0o750, \"others\", \"read\") and allowed(0o644, \"owner\", \"write\")\n    for bad in (\"rwx\", \"rwxrwxrwz\", \"xwrxwrxwr\"):\n        try:\n            from_symbolic(bad)\n            assert False, bad\n        except ValueError:\n            pass")

### Exercise: Packed colours

Write `pack(r, g, b)` and `unpack(colour)` for 24-bit colours (8 bits each; raise `ValueError` for a channel outside 0 to 255), `to_hex(colour)` giving `"#RRGGBB"` in capitals, and `blend(c1, c2)`: the colour whose channels are the averages (rounded down) of the two colours' channels.

In [ ]:
def pack(r, g, b):
    ___

def unpack(colour):
    ___

def to_hex(colour):
    ___

def blend(c1, c2):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_colours():\n    \"packing\"\n    assert pack(255, 136, 0) == 0xFF8800 and unpack(0xFF8800) == (255, 136, 0) and to_hex(0xFF8800) == \"#FF8800\"\n    assert to_hex(pack(0, 0, 10)) == \"#00000A\" and blend(0xFF0000, 0x0000FF) == pack(127, 0, 127)\n    for bad in ((256, 0, 0), (0, -1, 0)):\n        try:\n            pack(*bad)\n            assert False\n        except ValueError:\n            pass\n    rng = random.Random(2)\n    for _ in range(200):\n        c = tuple(rng.randint(0, 255) for _ in range(3))\n        assert unpack(pack(*c)) == c and int(to_hex(pack(*c))[1:], 16) == pack(*c)")

### Exercise: Fixed widths

Write `to_unsigned(x, bits)` and `to_signed(u, bits)` for two's complement in a given width, and `fits(x, bits)`: True if the whole number x can be stored as a **signed** `bits`-bit number (from −2^(bits−1) to 2^(bits−1) − 1). Then `add_wrapping(a, b, bits)`: the signed result of adding a and b in that width, wrapping around as hardware does (so in 8 bits, 127 + 1 is −128).

In [ ]:
def to_unsigned(x, bits):
    ___

def to_signed(u, bits):
    ___

def fits(x, bits):
    ___

def add_wrapping(a, b, bits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_widths():\n    \"two's complement\"\n    assert to_unsigned(-1, 8) == 255 and to_unsigned(-6, 8) == 250 and to_signed(250, 8) == -6 and to_signed(127, 8) == 127\n    assert to_signed(128, 8) == -128 and to_unsigned(-1, 32) == 0xFFFFFFFF\n    assert fits(127, 8) and not fits(128, 8) and fits(-128, 8) and not fits(-129, 8)\n    assert add_wrapping(127, 1, 8) == -128 and add_wrapping(-128, -1, 8) == 127 and add_wrapping(100, 27, 8) == 127\n    rng = random.Random(3)\n    for _ in range(300):\n        bits = rng.choice([4, 8, 16, 32])\n        x = rng.randint(-(1 << (bits - 1)), (1 << (bits - 1)) - 1)\n        assert to_signed(to_unsigned(x, bits), bits) == x and 0 <= to_unsigned(x, bits) < 1 << bits")

### Exercise: Counting the ones

Write `ones(n)`: the number of 1 bits in a non-negative whole number, using only `&`, `>>` and a loop (no `bin`, `bit_count` or strings). Then `parity(n)`: 1 if the number of 1 bits is odd, else 0, by XOR-ing the bits together.

In [ ]:
def ones(n):
    ___

def parity(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_ones():\n    \"ones\"\n    assert ones(0) == 0 and ones(0b1011) == 3 and ones(255) == 8 and ones(1 << 100) == 1\n    assert parity(0b1011) == 1 and parity(0b1001) == 0 and parity(0) == 0\n    rng = random.Random(4)\n    for _ in range(300):\n        n = rng.randint(0, 1 << 64)\n        assert ones(n) == bin(n).count(\"1\") and parity(n) == bin(n).count(\"1\") % 2\n    assert \"bin(\" not in source and \"bit_count\" not in source and \"str(\" not in source")

### Bug hunt: A mask that's one bit

The mask for the low 8 bits should be `(1 << 8) - 1`, eight ones. Without brackets, `1 << bits - 1` is `1 << 7`, a single bit, so this prints `128` instead of `250` for −6 in 8 bits. Add the brackets.

In [ ]:
def to_unsigned(x, bits):
    return x & 1 << bits - 1

print(to_unsigned(-6, 8))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"precedence\"\n    expect_output(\"250\")")

### Bug hunt: An inverted byte

To flip every bit of an 8-bit value, `~x` alone gives a negative Python integer (`~5` is `-6`), and this prints `-6` instead of `250`. Keep only the low 8 bits with a mask.

In [ ]:
x = 5
print(~x)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed width\"\n    expect_output(\"250\")")

### Bug hunt: Green in the wrong place

Unpacking a colour, green sits in bits 8 to 15. This shifts by 16 for green as well, so it prints red twice: `(255, 255, 0)`. Fix the shift so it prints `(255, 136, 0)`.

In [ ]:
colour = 0xFF8800
r = colour >> 16 & 0xFF
g = colour >> 16 & 0xFF
b = colour & 0xFF
print((r, g, b))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fields\"\n    expect_output(\"(255, 136, 0)\")")

### Puzzle: Setting, clearing and testing

Arrange the lines to set bit 3, clear bit 0, flip bit 1 and then list the bits that are on. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # 0b1100 ^ 0b0110, as a whole number
q = ___   # ~0
r = ___   # -1 & 0xF
s = ___   # (37).bit_length()
t = ___   # 0o750 >> 3 & 7

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10 and q == -1 and r == 15, \"p, q and r\"\n    assert s == 6 and t == 5, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a set in one number

Write a class `BitSet` that stores a set of small non-negative whole numbers in **one integer**, `self.bits`, with bit i set when i is in the set:

- `BitSet(items=())`; `add(i)`, `discard(i)`; `i in s` (`__contains__`); `len(s)` (count the 1s); iterating gives the members in increasing order (`__iter__` may return `iter` of a list); `ValueError` for a negative item.
- `s | t`, `s & t`, `s - t` and `s ^ t` returning new `BitSet`s, each a single integer operation; `s == t`; and `repr` like `BitSet({1, 4, 9})` (`BitSet()` when empty).
- `smallest()`: the smallest member, using `x & -x` to isolate the lowest set bit and `bit_length`; `ValueError` if empty.
- `complement(n)`: a new `BitSet` of the numbers 0 to n − 1 **not** in the set.

In [ ]:
class BitSet:
    def __init__(self, items=()):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_bitset():\n    \"a set in one number\"\n    s = BitSet([1, 4, 9])\n    assert s.bits == 0b1000010010 and 4 in s and 5 not in s and len(s) == 3 and list(s) == [1, 4, 9]\n    assert repr(s) == \"BitSet({1, 4, 9})\" and repr(BitSet()) == \"BitSet()\"\n    s.add(0)\n    s.discard(9)\n    s.discard(7)\n    assert list(s) == [0, 1, 4] and s.smallest() == 0\n    t = BitSet([4, 5])\n    assert list(s | t) == [0, 1, 4, 5] and list(s & t) == [4] and list(s - t) == [0, 1] and list(s ^ t) == [0, 1, 5]\n    assert BitSet([2, 3]) == BitSet([3, 2]) and BitSet([2]) != BitSet([3])\n    assert list(BitSet([1, 3]).complement(5)) == [0, 2, 4]\n    for bad in (lambda: BitSet([-1]), lambda: BitSet().smallest(), lambda: s.add(-3)):\n        try:\n            bad()\n            assert False\n        except ValueError:\n            pass\n    rng = random.Random(5)\n    for _ in range(200):\n        a = set(rng.sample(range(60), rng.randint(0, 20)))\n        b = set(rng.sample(range(60), rng.randint(0, 20)))\n        A, B = BitSet(a), BitSet(b)\n        assert list(A | B) == sorted(a | b) and list(A & B) == sorted(a & b)\n        assert list(A - B) == sorted(a - b) and list(A ^ B) == sorted(a ^ b) and len(A) == len(a)\n        if a:\n            assert A.smallest() == min(a)\n    assert len(BitSet(range(0, 10000, 3))) == 3334")

## Warm-up

`&` keeps bits set in **both**, `|` bits set in **either**, `^` bits set in **exactly one**. Shifting left by 3 multiplies by 2³; shifting right by 2 divides by 4 (rounding down). And `~5` is `-6`, which needs today's section on negative numbers to explain.

## Bases

A binary number's digits are worth 1, 2, 4, 8, ... from the right, just as decimal digits are worth 1, 10, 100. So `0b1011` is 8 + 2 + 1 = 11. **Octal** (base 8) groups bits in threes and **hexadecimal** (base 16, digits 0 to 9 then a to f) in fours, which is why both appear wherever bits do: one hex digit is exactly four bits, so a byte is two hex digits.

| | Write it | Read it | Show it |
|---|---|---|---|
| binary | `0b1011` | `int("1011", 2)` | `bin(11)`, `f"{11:b}"`, `f"{11:08b}"` |
| octal | `0o13` | `int("13", 8)` | `oct(11)`, `f"{11:o}"` |
| hex | `0xb` | `int("b", 16)` | `hex(11)`, `f"{11:x}"`, `f"{11:02X}"` |

The format specifications pad with zeros to a width (`08b` is eight binary digits). `int(text, 0)` reads a prefix (`0b`, `0o`, `0x`) to choose the base. `n.bit_length()` is the number of bits needed (the position of the highest 1, plus one), and `n.bit_count()` (Python 3.10 and later) counts the 1s.

## The six operators

| Operator | Bit i of the result is 1 when | Use it to |
|---|---|---|
| `a & b` | both have bit i | **test** or **keep** bits (a mask) |
| `a \| b` | either has bit i | **set** bits |
| `a ^ b` | exactly one has bit i | **flip** bits, compare |
| `~a` | a doesn't have bit i | **invert** (see below) |
| `a << k` | a has bit i − k | multiply by 2ᵏ |
| `a >> k` | a has bit i + k | floor-divide by 2ᵏ |

The patterns you'll use constantly, for bit i:

- test: `x >> i & 1` (or `x & (1 << i) != 0`, with brackets)
- set: `x | (1 << i)`; clear: `x & ~(1 << i)`; flip: `x ^ (1 << i)`

**Precedence trap**: arithmetic binds **more tightly** than shifts and the bitwise operators. So `1 << n - 1` means `1 << (n - 1)`, not `(1 << n) - 1`, and `x & 3 + 1` means `x & 4`. (Comparisons bind more loosely than all of them, so `x & 1 == 0` does mean `(x & 1) == 0` in Python, unlike in C, where it's a famous bug.) When in doubt, use brackets.

XOR has properties worth remembering: `x ^ x == 0`, `x ^ 0 == x`, and it's its own inverse: `(x ^ k) ^ k == x`. That makes it the basis of simple checksums, swapping without a temporary, and Day 2's "find the number without a partner".

Underscores in number literals (`0b1010_0110`) are ignored by Python and make long numbers easier to read.

## Shifts

`x << k` appends k zero bits: x × 2ᵏ. `x >> k` drops the k lowest bits: x // 2ᵏ (rounding **down**, so `-7 >> 1` is `-4`, like `-7 // 2`). Python integers have no fixed size, so left shifts never overflow: `1 << 1000` is a 302-digit number.

Shifts and masks together pick out a **field**: the bits from position lo upwards, k of them, are `x >> lo & ((1 << k) - 1)`. `(1 << k) - 1` is k ones, the standard mask.

## Negative numbers and fixed widths

Hardware stores integers in a fixed number of bits, and negative numbers in **two's complement**: in 8 bits, −1 is `11111111`, −2 is `11111110`, and −128 is `10000000`. The rule: −x is (~x) + 1, where ~ flips every bit. Adding works the same for positive and negative numbers, which is why hardware uses it.

Python integers have **no** fixed width, so Python behaves as if negative numbers had **infinitely many** leading 1s. That explains `~5 == -6` (flipping every bit of …000101 gives …111010, which is −6), and why `bin(-6)` shows `-0b110` instead of a row of ones. The identity to remember: `~x == -x - 1`.

To see or produce a **fixed-width** pattern (for a file format, a network protocol, a microcontroller), mask with `& ((1 << bits) - 1)`: `-6 & 0xFF` is 250, which is `11111010`. To go back, if the top bit is set, subtract 2^bits: 250 − 256 = −6.

## Fields and flags

**Flags**: each bit is a yes/no setting, and a whole set of settings is one integer. Unix file permissions are the familiar example: three groups (owner, group, others) of three bits (read 4, write 2, execute 1), usually written in octal: `0o750` is `rwxr-x---`, owner can do everything, the group can read and execute, others nothing. Python's `os.chmod` takes exactly these numbers.

**Packing**: several small numbers stored side by side in one integer. A colour in 24 bits holds red, green and blue, 8 bits each: `0xFF8800` is red 255, green 136, blue 0. Packing is shift and OR; unpacking is shift and mask.

## Practice

## Quiz

## Daily challenge

## Before you go

Write your age, your house number and today's date in binary and hex. Tomorrow: bit tricks, the small identities behind fast code, puzzles and hash functions.